# Week 4: Independent Evaluation, Human Audit & Final Benchmarking

**Reference:** *Data Extraction Attacks in Retrieval-Augmented Generation via Backdoors* (arXiv:2411.01705v2)  
**Hardware:** Single Kaggle T4 GPU (16 GB VRAM)  
**Scope:** Week 4 of 4: Running on-device Qwen2.5-7B judge (2,500 test queries), conducting 80-sample balanced human audit, measuring Cohen's Kappa, and producing final scientific benchmark tables and figures with 95% bootstrap confidence intervals.

---

### Step 0: Kaggle Setup & Hugging Face Authentication
Before running, ensure:
1. In the right panel, set **Accelerator** to **GPU T4 x1**.
2. In the right panel, set **Internet** to **On**.
3. Add your Hugging Face token under **Add-ons -> Secrets** as `HF_TOKEN`.

In [ ]:
# Ensure repository files are present
import os
if not os.path.exists('evaluation/on_device_judge.py'):
    print('Cloning repository into Kaggle working directory...')
    !git clone https://github.com/starboy1402/Rag_Backdoor.git /kaggle/working/Rag_Backdoor
    %cd /kaggle/working/Rag_Backdoor
else:
    print('Project files already present.')

try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret('HF_TOKEN')
    login(token=hf_token)
    print('Hugging Face authentication successful!')
except Exception as e:
    print(f'Hugging Face secret not found ({e}).')


### Step 1: On-Device Local LLM Judge Evaluation (Qwen2.5-7B 4-bit NF4)
Evaluate responses across all 5 test conditions (500 samples each = 2,500 base calls):
1. Paraphrase Triggered (Undefended)
2. Paraphrase Triggered (Privacy-Prompt Baseline)
3. Paraphrase Triggered (95% Entity Filter Baseline)
4. Paraphrase Triggered (Proposed Hybrid Detector)
5. Clean Non-Triggered (Benign False Positive Rate)
Borderline responses trigger a 3-stochastic-call tie-breaker at temperature 0.3.

In [ ]:
# Run on-device local LLM judge
!python evaluation/on_device_judge.py \
    --test-file cache/medmcqa_test_500.json \
    --retrieval-cache cache/retrieval_cache.json \
    --output-file cache/judge_evaluations.json \
    --use-gpu


### Step 2: Balanced 80-Sample Human Audit & Judge Validation
Extract 40 predicted LEAK and 40 predicted BENIGN outputs, strip all model tags, and evaluate:
- Inter-annotator agreement (Cohen's Kappa $\kappa_{\text{human-human}}$)
- Qwen judge agreement against consensus human ground truth (target $\kappa_{\text{judge-human}} \ge 0.70$)
- Judge Precision, Recall, and F1

In [ ]:
# Generate blind audit sheet and compute validation metrics
!python evaluation/human_audit.py \
    --eval-file cache/judge_evaluations.json \
    --out-csv cache/human_audit_sheet_80.csv \
    --run-mock-validation


### Step 3: Final Comprehensive Benchmark Table
Compile all metrics across conditions with 95% bootstrap confidence intervals (1,000 resamples).

In [ ]:
# Generate complete benchmark summary
!python evaluation/evaluate_metrics.py \
    --test-file cache/medmcqa_test_500.json \
    --retrieval-cache cache/retrieval_cache.json \
    --clean-model ./checkpoints/gemma_2b_clean_baseline \
    --paraphrase-model ./checkpoints/gemma_2b_paraphrase_5pct \
    --detector-bundle cache/detector_bundle.pkl \
    --output-file cache/final_research_benchmark.json


### Step 4: Display Formatted Research Results
Print scientific summary table for inclusion in research paper and presentation deck.

In [ ]:
import json
with open('cache/final_research_benchmark.json', 'r') as f:
    res = json.load(f)

print("=" * 60)
print("         FINAL RAG BACKDOOR DEFENSE BENCHMARK RESULTS       ")
print("=" * 60)
for k, v in res.items():
    print(f"{k:<32}: {v}")
print("=" * 60)
